In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import os

2025-01-22 00:44:43.528157: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1737503083.597916  100435 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1737503083.620188  100435 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-01-22 00:44:43.791237: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
mp_face_mesh = mp.solutions.face_mesh
mp_hands = mp.solutions.hands
mp_pose = mp.solutions.pose

In [6]:
def calculate_distance(point1, point2):
    """Calculate Euclidean distance between two points."""
    return np.linalg.norm(np.array(point1) - np.array(point2))

def setup_directory(path):
    """Ensure the directory exists."""
    if not os.path.exists(path):
        os.makedirs(path)
        
def calculate_angle_between_vectors(v1, v2):
    """Calculate the angle in degrees between two vectors."""
    unit_vector_1 = v1 / np.linalg.norm(v1)
    unit_vector_2 = v2 / np.linalg.norm(v2)
    dot_product = np.dot(unit_vector_1, unit_vector_2)
    angle = np.arccos(dot_product)
    return np.degrees(angle)

def calculate_gaze_angles(eye_left, eye_right, nose_tip, frame):
    """Compute horizontal angle (degrees) and vertical offset in pixels."""
    x_left,  y_left  = eye_left.x  * frame.shape[1], eye_left.y  * frame.shape[0]
    x_right, y_right = eye_right.x * frame.shape[1], eye_right.y * frame.shape[0]
    x_nose,  y_nose  = nose_tip.x  * frame.shape[1], nose_tip.y  * frame.shape[0]

    # Midpoint
    mid_x = (x_left + x_right) / 2
    mid_y = (y_left + y_right) / 2

    dx = x_nose - mid_x
    dy = y_nose - mid_y

    horizontal_angle = np.degrees(np.arctan2(dy, dx))
    vertical_offset  = dy  # in pixels

    return horizontal_angle, vertical_offset



csv_files = ['./output-csv/video_8.csv']
output_directory = './features-new/'
setup_directory(output_directory)

In [ ]:
for csv_file in csv_files:
    video_file = csv_file.replace('.csv', '.mp4')
    annotations = pd.read_csv(csv_file)
    cap = cv2.VideoCapture(video_file)
    
    prev_wrist_distance = [None, None]
    
    frame_rate = cap.get(cv2.CAP_PROP_FPS)
    output_features = []

    with mp_face_mesh.FaceMesh(static_image_mode=True, max_num_faces=2, min_detection_confidence=0.3) as face_mesh, \
         mp_hands.Hands(static_image_mode=False, max_num_hands=4, min_detection_confidence=0.5) as hands, \
         mp_pose.Pose(static_image_mode=False, min_detection_confidence=0.5) as pose:

        frame_index = 0
        while cap.isOpened():
            success, frame = cap.read()
            if not success:
                break

            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            face_results = face_mesh.process(frame_rgb)
            hand_results = hands.process(frame_rgb)
            pose_results = pose.process(frame_rgb)
            current_time = frame_index / frame_rate * 1000

            for idx in range(2):
                side = 'Left' if idx == 0 else 'Right'

                # CHANGED: Initialize binary columns to 0 instead of "no_X"
                frame_features = {
                    'video number': video_file.split('/')[-1].replace('.mp4', ''),
                    'frame number': frame_index,
                    'mouth_width': None,
                    'lip_distance': None,
                    'wrist_to_mouth_distance': None,
                    'wrist_to_mouth_velocity': 0.0,
                    'eyebrow_distance': None,
                    'left_ear_to_lip': None,
                    'right_ear_to_lip': None,
                    'jaw_to_nose': None,
                    'vertical_face_angle': None,
                    'vertical_gaze_angle': None,
                    'horizontal_gaze_angle': None,

                    # Five binary columns
                    'speaking': 0,
                    'eating': 0,
                    'smiling': 0,
                    'fork_towards': 0,
                    'action': 0
                }

                face_landmarks = (face_results.multi_face_landmarks[idx]
                                  if face_results.multi_face_landmarks and idx < len(face_results.multi_face_landmarks)
                                  else None)
                pose_landmarks = pose_results.pose_landmarks

                if face_landmarks:
                    # 1) Calculate eye distance for ratio-based normalization
                    eye_left = face_landmarks.landmark[133]
                    eye_right = face_landmarks.landmark[362]
                    eye_distance = calculate_distance(
                        (eye_left.x * frame.shape[1], eye_left.y * frame.shape[0]),
                        (eye_right.x * frame.shape[1], eye_right.y * frame.shape[0])
                    )
                    # Avoid zero eye distance
                    if eye_distance < 1e-6:
                        eye_distance = 1e-6

                    # 2) Other landmarks
                    mouth_left = face_landmarks.landmark[61]
                    mouth_right = face_landmarks.landmark[291]
                    upper_lip = face_landmarks.landmark[13]
                    lower_lip = face_landmarks.landmark[14]
                    left_ear = face_landmarks.landmark[234]
                    right_ear = face_landmarks.landmark[454]
                    jaw = face_landmarks.landmark[152]
                    eyebrow_left = face_landmarks.landmark[55]
                    eyebrow_right = face_landmarks.landmark[285]
                    nose_tip = face_landmarks.landmark[1]

                    # 3) Convert key coordinates to (x, y)
                    mouth_x = (upper_lip.x + lower_lip.x) / 2
                    mouth_y = (upper_lip.y + lower_lip.y) / 2
                    mouth_point = (mouth_x * frame.shape[1], mouth_y * frame.shape[0])
                    left_ear_point = (left_ear.x * frame.shape[1], left_ear.y * frame.shape[0])
                    right_ear_point = (right_ear.x * frame.shape[1], right_ear.y * frame.shape[0])
                    jaw_point = (jaw.x * frame.shape[1], jaw.y * frame.shape[0])
                    nose_point = (nose_tip.x * frame.shape[1], nose_tip.y * frame.shape[0])

                    # 4) Distances & Angles
                    mouth_width = calculate_distance(
                        (mouth_left.x * frame.shape[1], mouth_left.y * frame.shape[0]),
                        (mouth_right.x * frame.shape[1], mouth_right.y * frame.shape[0])
                    )
                    lip_distance = calculate_distance(
                        (upper_lip.x * frame.shape[1], upper_lip.y * frame.shape[0]),
                        (lower_lip.x * frame.shape[1], lower_lip.y * frame.shape[0])
                    )
                    left_ear_to_lip_distance = calculate_distance(left_ear_point, mouth_point)
                    right_ear_to_lip_distance = calculate_distance(right_ear_point, mouth_point)
                    jaw_to_nose_distance = calculate_distance(jaw_point, nose_point)
                    eyebrow_distance = calculate_distance(
                        (eyebrow_left.x * frame.shape[1], eyebrow_left.y * frame.shape[0]),
                        (eyebrow_right.x * frame.shape[1], eyebrow_right.y * frame.shape[0])
                    )

                    # Face angle
                    vertical_vector = np.array([jaw.x - nose_tip.x, 
                                                jaw.y - nose_tip.y, 
                                                jaw.z - nose_tip.z])
                    reference_vertical = np.array([0, 0, 1])
                    vertical_face_angle = calculate_angle_between_vectors(vertical_vector, reference_vertical)

                    # Gaze angles
                    horizontal_gaze_angle, vertical_gaze_angle = calculate_gaze_angles(
                        eye_left, eye_right, nose_tip, frame
                    )

                    # 5) Normalize each distance by eye_distance
                    #    So if eye_distance ~ 100 px, mouth_width is e.g. mouth_width_px / 100
                    mouth_width_ratio            = mouth_width / eye_distance
                    lip_distance_ratio           = lip_distance / eye_distance
                    left_ear_to_lip_ratio        = left_ear_to_lip_distance / eye_distance
                    right_ear_to_lip_ratio       = right_ear_to_lip_distance / eye_distance
                    jaw_to_nose_ratio            = jaw_to_nose_distance / eye_distance
                    eyebrow_distance_ratio       = eyebrow_distance / eye_distance

                    # For angles, keep them in degrees or offsets. We'll standardize later anyway.
                    frame_features.update({
                        'mouth_width': mouth_width_ratio,
                        'lip_distance': lip_distance_ratio,
                        'left_ear_to_lip': left_ear_to_lip_ratio,
                        'right_ear_to_lip': right_ear_to_lip_ratio,
                        'jaw_to_nose': jaw_to_nose_ratio,
                        'eyebrow_distance': eyebrow_distance_ratio,
                        'vertical_face_angle': vertical_face_angle,      # degrees
                        'vertical_gaze_angle': vertical_gaze_angle,      # pixel offset
                        'horizontal_gaze_angle': horizontal_gaze_angle,  # degrees
                    })

                if pose_landmarks and face_landmarks:
                    # Identify which wrist to use based on side
                    if side == 'Left':
                        wrist_landmark = mp_pose.PoseLandmark.RIGHT_WRIST
                    else:
                        wrist_landmark = mp_pose.PoseLandmark.LEFT_WRIST

                    wrist = pose_landmarks.landmark[wrist_landmark]
                    wrist_point = (int(wrist.x * frame.shape[1]), int(wrist.y * frame.shape[0]))

                    # If we have mouth_point from above
                    # (Check if mouth_point is defined — if face_landmarks is None, skip)
                    if 'mouth_point' in locals():
                        wrist_to_mouth_distance = calculate_distance(wrist_point, mouth_point)
                        frame_features['wrist_to_mouth_distance'] = wrist_to_mouth_distance

                        prev_dist = prev_wrist_distance[idx]
                        if prev_dist is not None:
                            velocity = (wrist_to_mouth_distance - prev_dist) * frame_rate
                        else:
                            velocity = 0.0
                        frame_features['wrist_to_mouth_velocity'] = velocity
                        prev_wrist_distance[idx] = wrist_to_mouth_distance
                    
                    
                # CHANGED: Set binary columns based on annotation
                for _, row in annotations.iterrows():
                    if row['Start Time'] <= current_time <= row['End Time'] and side in row['Annotation']:
                        current_action = row['Annotation'].replace(f'{side}_', '')
                        if current_action in ['speaking', 'eating', 'smiling', 'Fork_towards']:
                            if current_action == 'Fork_towards':
                                frame_features['fork_towards'] = 1
                            else:
                                frame_features[current_action] = 1
                            frame_features['action'] = 1

                output_features.append(frame_features)
            frame_index += 1

    cap.release()

    # CHANGED: New columns reflect binary labels (no more "no_speaking," etc.)
    columns = [
        'video number', 'frame number',
        'mouth_width', 'lip_distance', 'wrist_to_mouth_distance', 'wrist_to_mouth_velocity',
        'left_ear_to_lip', 'right_ear_to_lip', 'jaw_to_nose', 'eyebrow_distance',
        'vertical_face_angle', 'vertical_gaze_angle', 'horizontal_gaze_angle',
        'speaking', 'eating', 'smiling', 'fork_towards', 'action'
    ]
    features_df = pd.DataFrame(output_features, columns=columns)
    
    if 'wrist_to_mouth_velocity' not in features_df.columns:
        features_df['wrist_to_mouth_velocity'] = 0.0

    features_df['wrist_to_mouth_velocity'] = (
        features_df['wrist_to_mouth_velocity']
        .rolling(window=5, min_periods=1)
        .mean()
    )

    numeric_cols = [
        'mouth_width', 'lip_distance', 'wrist_to_mouth_distance',
        'wrist_to_mouth_velocity', 'left_ear_to_lip', 'right_ear_to_lip',
        'jaw_to_nose', 'eyebrow_distance', 'vertical_face_angle',
        'vertical_gaze_angle', 'horizontal_gaze_angle'
    ]

    for col in numeric_cols:
        mean_val = features_df[col].mean()
        std_val = features_df[col].std()
        if std_val != 0:
            features_df[col] = (features_df[col] - mean_val) / std_val
        else:
            features_df[col] = 0.0

    # ------------------------------------------------
    # Save CSV
    # ------------------------------------------------
    output_csv = os.path.join(
        output_directory, 
        f"{video_file.split('/')[-1].replace('.mp4', '')}_features.csv"
    )
    features_df.to_csv(output_csv, index=False)
    print(f"Feature extraction complete. Saved to {output_csv}")

In [9]:
import pandas as pd

df = pd.read_csv('./features-new/video_4_features.csv')
print(df['speaking'].value_counts())
print(df['eating'].value_counts())
print(df['smiling'].value_counts())
print(df['fork_towards'].value_counts())
print(df['action'].value_counts())

speaking
no_speaking    20675
speaking       12185
Name: count, dtype: int64
eating
no_eating    24802
eating        8058
Name: count, dtype: int64
smiling
no_smiling    31555
smiling        1305
Name: count, dtype: int64
Fork_towards
no_Fork_towards    30409
Fork_towards        2451
Name: count, dtype: int64
action
action       31310
no_action     1550
Name: count, dtype: int64


In [2]:
import pandas as pd

# Load the Excel file without converting "NA" to NaN
file_path = "20240516_training_data.xlsx"
df = pd.read_excel(file_path, keep_default_na=False)

# Standardize column names (remove leading/trailing spaces)
df.columns = df.columns.str.strip()

# Convert "PRENOTAZIONE" to string to avoid data type mismatches
df["PRENOTAZIONE"] = df["PRENOTAZIONE"].astype(str)

# Define the values to find (as strings)
values_to_find = ["818941", "822959", "823362", "823610", "849245"]

# Filter the dataframe
filtered_df = df[df["PRENOTAZIONE"].isin(values_to_find)]

# Save to a new CSV file
output_file = "filtered_data.csv"
filtered_df.to_csv(output_file, index=False, na_rep="")

print(f"Filtered data saved to {output_file}")

Filtered data saved to filtered_data.csv


In [2]:
import pandas as pd

# Define file paths
csv_file = "filtered_data.csv"  # Replace with your CSV file name
xlsx_file = "output.xlsx"  # Replace with your desired output file name

# Read the CSV file while keeping "NA" as a string
df = pd.read_csv(csv_file, keep_default_na=False)

# Save it as an Excel file
df.to_excel(xlsx_file, index=False, engine='openpyxl')

print(f"Conversion successful! Saved as {xlsx_file}")


Conversion successful! Saved as output.xlsx
